# BM25 and the Binary Independence Model — companion notebook

This notebook is the **narrative** half of the BM25 Python pillar. The reusable,
tested implementation lives next door in [`bm25.py`](bm25.py); here we import it
and walk the topic section by section, so every claim the page makes renders as
executed output.

**Two artifacts, one source of truth.** `bm25.py` owns the numbers — its
`assert`-based harness encodes the limit theorems and the length-hijack flip, and
its finance corpus is mirrored *to the decimal* by `BM25ScoringLaboratory.tsx`.
This notebook never redefines that math; it calls into it. Change a number in
`bm25.py`, then re-run this notebook (and the viz) — never the other way around.

Run from the repo root:

```
uv run --with numpy --with scipy --with rank-bm25 --with jupyter \
    jupyter execute notebooks/bm25/01_bm25.ipynb
```

or run the reference implementation directly, which executes the same checks:

```
uv run --with numpy --with scipy --with rank-bm25 python notebooks/bm25/bm25.py
```

In [ ]:
import sys
import pathlib

# Make bm25.py importable whether the kernel starts in the repo root or in this
# notebook's own directory.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "bm25", pathlib.Path("notebooks/bm25")):
    if (_cand / "bm25.py").exists():
        sys.path.insert(0, str(_cand))
        break

from bm25 import (
    build_inverted_index, idf_scalar, idf_vector, tf_factor,
    bm25_contributions, bm25_rank, ndcg_at_k,
    _FINANCE_CORPUS, _QUERY, _QRELS,
    test_limits, test_monotonicity, test_length_hijack,
    validate_against_rank_bm25, finance_demo, grid_sweep,
)

## 1. From term counts to a ranking: the corpus and inverted index

BM25 turns a query and a document into a score by summing per-term weights over an
inverted index. We build the index over the finance corpus and read off the three
quantities the formula needs: the document frequency $df_t$ of each query term, the
document lengths, and the average length $\mathrm{avgdl}$. The padded earnings-call
transcript is deliberately long — that length is what the normalization term will
later have to fight.

In [ ]:
index = build_inverted_index(_FINANCE_CORPUS)
print(f"query = {_QUERY!r}")
print(f"N     = {index.n_docs} documents")
print(f"avgdl = {index.avgdl:.2f} tokens")
print()
print(f"{'document':<18}{'length':>8}")
for doc_id, dl in zip(index.doc_ids, index.doc_len):
    print(f"{doc_id:<18}{int(dl):>8}")
print()
print("document frequency of each query term:")
for term in _QUERY.split():
    print(f"  df[{term:>9}] = {int(index.df[index.vocab[term]])}")

## 2. IDF emerges from probability theory

The Robertson–Spärck-Jones relevance weight, under a Jeffreys $(+0.5)$ prior and
with no relevance judgments, collapses to a smoothed inverse document frequency. We
print the BM25 IDF beside the textbook $\log(N/df)$ so the smoothing is visible: the
$+0.5$ continuity correction keeps the weight finite, and rare terms (small $df$)
earn the largest weight.

In [ ]:
N = 1000
print(f"{'df':>5}{'bm25 IDF':>12}{'textbook IDF':>15}")
for df in (1, 10, 100, 500, 999):
    print(f"{df:>5}{idf_scalar(df, N, 'bm25'):>12.3f}{idf_scalar(df, N, 'textbook'):>15.3f}")

## 3. Eliteness and saturation: the term-frequency factor

The 2-Poisson eliteness model motivates a *saturating* transform of raw term
frequency: each additional occurrence adds less than the last, and the factor
approaches a ceiling of $k_1 + 1$. The $b$ parameter interpolates document-length
normalization — at $b = 0$ length is ignored; at $b = 1$ a document twice the average
length has its term counts discounted toward half.

In [ ]:
k1 = 1.5
print(f"saturation toward the ceiling k1 + 1 = {k1 + 1:.1f}  (dl = avgdl, b = 0.75):")
print(f"{'tf':>6}{'tf_factor':>12}")
for tf in (1, 2, 3, 5, 10, 50, 1000):
    print(f"{tf:>6}{tf_factor(float(tf), k1, 0.75, 80.0, 80.0):>12.3f}")

print()
print("length normalization (tf = 5, k1 = 1.5, avgdl = 80):")
print(f"{'b':>5}{'short dl=30':>14}{'long dl=300':>14}")
for b in (0.0, 0.5, 1.0):
    short = tf_factor(5.0, k1, b, 30.0, 80.0)
    longd = tf_factor(5.0, k1, b, 300.0, 80.0)
    print(f"{b:>5}{short:>14.3f}{longd:>14.3f}")

## 4. The limit theorems, as executable checks

Four limits pin down the formula's behavior, and all four are asserted in
`bm25.py`: $k_1 \to 0$ recovers the binary BIM (factor $\to 1$), $k_1 \to \infty$
recovers length-normalized raw $tf$, the factor saturates at $k_1 + 1$, and $b$
toggles length normalization. We also assert the factor is monotonically increasing
in $tf$. A green line here means the topic's Theorem 2 holds numerically.

In [ ]:
test_limits()
test_monotonicity()

## 5. Assembling BM25: per-term contributions and the ranking

A document's score is the sum of $\mathrm{IDF}_t \times \mathrm{tf\text{-}factor}_t$
over the query terms it contains. We read off the per-term contributions for the
concise on-point filing, then rank the whole corpus at the canonical
$(k_1, b) = (1.5, 0.75)$.

In [ ]:
idf_vec = idf_vector(index, "bm25")
doc_i = index.doc_ids.index("filing-onpoint")
print("per-term BM25 contributions for 'filing-onpoint' (k1=1.5, b=0.75):")
for term, contrib in bm25_contributions(_QUERY, doc_i, index, idf_vec).items():
    print(f"  {term:>9}: {contrib:.3f}")
print()
print("full ranking at (k1, b) = (1.5, 0.75):")
for rank, (doc_id, score) in enumerate(bm25_rank(_QUERY, index, k1=1.5, b=0.75), 1):
    print(f"  {rank}. {doc_id:<18} {score:.3f}")

## 6. The finance length-hijack and the $(k_1, b)$ sweep

At $b = 0$ the padded earnings-call transcript hijacks the top spot on raw counts
alone; at $b = 0.75$ length normalization demotes it and surfaces the concise 10-K
disclosure. This flip is the topic's central worked example — and an `assert` in
`bm25.py`. The grid sweep then finds the $(k_1, b)$ maximizing NDCG@10 against the
relevance judgments `_QRELS`.

In [ ]:
test_length_hijack()
print()
finance_demo()
print()
grid_sweep()

## 7. Cross-validation against `rank_bm25`

Finally we check the full ranking against the `rank_bm25` library's `BM25Okapi`,
matching its IDF variant (Robertson-floored `okapi`) so any residual difference
would be a genuine scoring bug rather than the documented difference between the
Robertson-floored and Lucene IDF forms. If `rank_bm25` is not installed the check
prints a skip line instead of failing.

In [ ]:
validate_against_rank_bm25(_FINANCE_CORPUS, _QUERY)

---

Every numerical claim above is also an `assert` in [`bm25.py`](bm25.py), so running
that file as a script is a regression test for the whole topic:

```
uv run --with numpy --with scipy --with rank-bm25 python notebooks/bm25/bm25.py
```

and `BM25ScoringLaboratory.tsx` reproduces this exact corpus and the same
$b = 0 \to$ padded-transcript-#1, $b = 0.75 \to$ on-point-filing-#1 flip. The three
pillars — math, viz, and code — agree by construction.